# Triadic closure in the email-Eu-core network — full reproduction code
Reproduces every number and figure in the paper: audit of the milestone figures, exact wedge enumeration, department comparison, degree-stratified analysis (Exp. 1), label-permutation test (Exp. 2), and degree-preserving randomization (Exp. 3).

**Setup:** run the first cell, then upload `email-Eu-core.txt` and `email-Eu-core-department-labels.txt` when prompted (or place them in `/content`).
**Runtime:** about 4–6 minutes with the default settings (most of it is Exp. 3).

In [1]:
# 1. Setup and parameters
import networkx as nx, numpy as np, pandas as pd, json, random, time, os
import matplotlib.pyplot as plt
from math import comb
plt.rcParams.update({'font.size':9,'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False})

N_PERM      = 2000   # label permutations (Exp. 2)
PERM_SEED   = 2026
N_RANDOM    = 60     # degree-preserving randomized graphs (Exp. 3); raise to 500+ for tighter p
SWAP_FACTOR = 10     # swaps per graph = SWAP_FACTOR * m
SWAP_SEED   = 7
N_BOOT      = 2000   # cluster-bootstrap resamples
BOOT_SEED   = 1

try:
    from google.colab import files
    if not (os.path.exists('email-Eu-core.txt') and os.path.exists('email-Eu-core-department-labels.txt')):
        files.upload()
except ImportError:
    pass

KeyboardInterrupt: 

## 2. Load data and preprocess (self-loops, direction, duplicates)

In [ ]:
edges = pd.read_csv('email-Eu-core.txt', sep=' ', header=None, names=['s','d'])
lab   = pd.read_csv('email-Eu-core-department-labels.txt', sep=' ', header=None, names=['n','dep'])
dept = np.zeros(1005, dtype=int); dept[lab.n.values] = lab.dep.values

R = {}
R['raw_directed_rows']   = len(edges)
R['selfloops_rows']      = int((edges.s==edges.d).sum())
R['unique_directed_pairs'] = int(len(edges[edges.s!=edges.d].drop_duplicates()))

G = nx.from_pandas_edgelist(edges, 's', 'd', create_using=nx.Graph())
G.remove_edges_from(list(nx.selfloop_edges(G)))
n, m = G.number_of_nodes(), G.number_of_edges()

D = nx.from_pandas_edgelist(edges[edges.s!=edges.d], 's', 'd', create_using=nx.DiGraph())
R['reciprocity'] = nx.reciprocity(D)

deg = np.array([G.degree(i) for i in range(n)])
R.update(n=n, m=m, density=nx.density(G), mean_deg=deg.mean(), median_deg=float(np.median(deg)),
         max_deg=int(deg.max()), n_deg0=int((deg==0).sum()), n_deg1=int((deg==1).sum()),
         n_deg_lt10=int((deg<10).sum()), n_deg_lt2=int((deg<2).sum()),
         lcc=len(max(nx.connected_components(G), key=len)), n_depts=int(len(set(dept))))
print({k:R[k] for k in ['raw_directed_rows','selfloops_rows','unique_directed_pairs','reciprocity','n','m','density','lcc','n_depts']})

## 3. Clustering statistics and internal-consistency check

In [ ]:
tri = sum(nx.triangles(G).values())//3
cc  = nx.clustering(G); cc_arr = np.array([cc[i] for i in range(n)])
R['triangles'] = tri
R['transitivity'] = nx.transitivity(G); R['avg_clust'] = nx.average_clustering(G)
R['n_wedges'] = int(sum(comb(int(k),2) for k in deg))
R['transitivity_check'] = 3*tri/R['n_wedges']           # must equal transitivity
R['n_cc0'] = int((cc_arr==0).sum()); R['n_cc1'] = int((cc_arr==1).sum())
R['avg_clust_excl_deg_lt2'] = float(cc_arr[deg>=2].mean())   # = center-uniform closure P_C
for k in ['triangles','transitivity','avg_clust','n_wedges','transitivity_check','n_cc0','n_cc1','avg_clust_excl_deg_lt2']:
    print(k, R[k])
print('ER ratio:', R['transitivity']/R['density'])

## 4. Enumerate ALL wedges: uniform vs center-uniform closure (resolves 0.2674 vs 0.45)

In [ ]:
adj = {i: sorted(G[i]) for i in range(n)}
A=[];C=[];B=[]
for b in range(n):
    nb = adj[b]
    for i in range(len(nb)):
        for j in range(i+1, len(nb)):
            A.append(nb[i]); C.append(nb[j]); B.append(b)
A=np.array(A); C=np.array(C); B=np.array(B)
AM = np.zeros((n,n), dtype=bool)
for u,v in G.edges(): AM[u,v]=AM[v,u]=True
CL = AM[A,C]                                  # closed flag per wedge
assert len(A)==R['n_wedges']

R['uniform_closure'] = float(CL.mean())        # = transitivity
w = 1.0/(deg[B]*(deg[B]-1)/2)                  # center-uniform weights
nc = int((deg>=2).sum())
R['center_uniform_closure_exact'] = float((w*CL).sum()/nc)
print('uniform  P_U =', R['uniform_closure'])
print('center-uniform P_C =', R['center_uniform_closure_exact'])

# Reproduce the milestone sampler (center-uniform, seed 42) and a uniform-wedge sample
def milestone_sampler(seed=42, N=50000):
    rng = random.Random(seed); nodes=[x for x in G.nodes() if G.degree(x)>=2]
    t=cl=st=sc=dt=dc=0
    for _ in range(N):
        b=rng.choice(nodes); a,c=rng.sample(list(G.neighbors(b)),2)
        x=G.has_edge(a,c); t+=1; cl+=x
        if dept[a]==dept[c]: st+=1; sc+=x
        else: dt+=1; dc+=x
    return cl/t, st, sc/st, dt, dc/dt
R['nb_sample_seed42'] = milestone_sampler()
R['uniform_sample_closure'] = float(CL[np.random.default_rng(42).integers(0,len(A),50000)].mean())
print('milestone sampler:', R['nb_sample_seed42'])
print('uniform-wedge sample (50k):', R['uniform_sample_closure'])

## 5. Department comparison (exact), assortativity, baseline tie density, cluster bootstrap

In [ ]:
same = dept[A]==dept[C]
def stat(s, cl, wt=None):
    if wt is None:
        return cl[s].mean()-cl[~s].mean(), cl[s].mean(), cl[~s].mean()
    ps=(wt[s]*cl[s]).sum()/wt[s].sum(); pd_=(wt[~s]*cl[~s]).sum()/wt[~s].sum()
    return ps-pd_, ps, pd_

d,ps,pdf = stat(same,CL)
R.update(U_same_n=int(same.sum()), U_diff_n=int((~same).sum()),
         U_same_closed=int(CL[same].sum()), U_diff_closed=int(CL[~same].sum()),
         U_same=ps, U_diff=pdf, U_diff_stat=d)
dc,psc,pdc = stat(same,CL,w)
R.update(C_same=psc, C_diff=pdc, C_diff_stat=dc, C_same_share=float(w[same].sum()/w.sum()))
print('Uniform  : same %.4f diff %.4f gap %.4f'%(ps,pdf,d))
print('Center-un: same %.4f diff %.4f gap %.4f'%(psc,pdc,dc))
print('share of wedges same-dept: uniform %.3f, center-uniform %.3f'%(same.mean(), R['C_same_share']))

# homophily in tie formation (baseline for closure)
nx.set_node_attributes(G, {i:int(dept[i]) for i in range(n)}, 'dep')
R['dept_assort'] = nx.attribute_assortativity_coefficient(G,'dep')
R['edge_same_dept_frac'] = float(np.mean([dept[u]==dept[v] for u,v in G.edges()]))
sizes = np.bincount(dept); R['pair_same_dept_frac_random'] = float((sizes*(sizes-1)).sum()/(n*(n-1)))
iu = np.triu_indices(n,1); sp = dept[iu[0]]==dept[iu[1]]
R['edge_density_same']=float(AM[iu][sp].mean()); R['edge_density_diff']=float(AM[iu][~sp].mean())
print('assortativity', R['dept_assort'], '| edges within dept', R['edge_same_dept_frac'], 'vs random', R['pair_same_dept_frac_random'])
print('pair density same/diff', R['edge_density_same'], R['edge_density_diff'])
print('closure lift over baseline: same %.2f, diff %.2f'%(ps/R['edge_density_same'], pdf/R['edge_density_diff']))

# naive Wald CI on exact counts
se = np.sqrt(ps*(1-ps)/R['U_same_n'] + pdf*(1-pdf)/R['U_diff_n'])
print('naive 95%% CI of gap: [%.4f, %.4f]'%(d-1.96*se, d+1.96*se))

# cluster bootstrap over centre nodes
g = pd.DataFrame({'b':B,'s':same,'c':CL})
ns = g[g.s].groupby('b').size().reindex(range(n),fill_value=0).values
nd = g[~g.s].groupby('b').size().reindex(range(n),fill_value=0).values
cs = g[g.s].groupby('b').c.sum().reindex(range(n),fill_value=0).values
cd = g[~g.s].groupby('b').c.sum().reindex(range(n),fill_value=0).values
keep = (ns+nd)>0; ns,nd,cs,cd = ns[keep],nd[keep],cs[keep],cd[keep]
rb = np.random.default_rng(BOOT_SEED); bs=[]
for _ in range(N_BOOT):
    ii = rb.integers(0,len(ns),len(ns))
    bs.append(cs[ii].sum()/ns[ii].sum() - cd[ii].sum()/nd[ii].sum())
R['cluster_boot_ci'] = [float(np.percentile(bs,2.5)), float(np.percentile(bs,97.5))]
print('cluster-bootstrap 95%% CI: [%.4f, %.4f]'%tuple(R['cluster_boot_ci']))

## 6. Experiment 1 — degree-stratified closure
Strata are fixed cut points. Wedges are assigned by (a) the centre's degree and (b) the smaller outer-node degree.

In [ ]:
def strat(key, bins, names):
    out=[]
    for lo,hi,nm in zip(bins[:-1],bins[1:],names):
        mk=(key>=lo)&(key<hi); s=mk&same; dd=mk&~same
        out.append(dict(group=nm, wedges=int(mk.sum()),
            same_n=int(s.sum()), same_closed=int(CL[s].sum()), same_open=int(s.sum()-CL[s].sum()),
            diff_n=int(dd.sum()), diff_closed=int(CL[dd].sum()), diff_open=int(dd.sum()-CL[dd].sum()),
            same_rate=float(CL[s].mean()) if s.sum() else None,
            diff_rate=float(CL[dd].mean()) if dd.sum() else None))
    return out

cb=[2,11,26,51,101,10**6]; cn=['2-10','11-25','26-50','51-100','>100']
R['strat_center']       = strat(deg[B], cb, cn)
R['strat_center_nodes'] = [int(((deg>=lo)&(deg<hi)).sum()) for lo,hi in zip(cb[:-1],cb[1:])]
mo = np.minimum(deg[A],deg[C])
R['strat_minouter'] = strat(mo, [1,6,11,26,51,10**6], ['1-5','6-10','11-25','26-50','>50'])

def std_diff(rows):
    rows=[x for x in rows if x['same_n'] and x['diff_n']]; tot=sum(x['wedges'] for x in rows)
    return sum(x['wedges']/tot*(x['same_rate']-x['diff_rate']) for x in rows)
R['std_diff_center'], R['std_diff_minouter'] = std_diff(R['strat_center']), std_diff(R['strat_minouter'])

for key in ['strat_center','strat_minouter']:
    print(pd.DataFrame(R[key]).assign(gap=lambda t:t.same_rate-t.diff_rate).round(3).to_string(index=False), '\n')
print('standardized gap (centre degree): %.3f | (min outer degree): %.3f'%(R['std_diff_center'],R['std_diff_minouter']))

## 7. Experiment 2 — department-label permutation test
Labels are permuted among nodes (department sizes preserved); the statistic is recomputed exactly on all wedges. Two-sided p = (1 + #{|Δ_b| ≥ |Δ_obs|}) / (B + 1).

In [ ]:
def run_perm(Bn=N_PERM, seed=PERM_SEED):
    rng=np.random.default_rng(seed)
    ob=stat(same,CL)[0]; obc=stat(same,CL,w)[0]; nu=[]; ncu=[]
    for _ in range(Bn):
        p=rng.permutation(dept); s=p[A]==p[C]
        nu.append(CL[s].mean()-CL[~s].mean())
        ps_=(w[s]*CL[s]).sum()/w[s].sum(); pd_=(w[~s]*CL[~s]).sum()/w[~s].sum(); ncu.append(ps_-pd_)
    nu=np.array(nu); ncu=np.array(ncu)
    pv=lambda o,x:(1+(np.abs(x)>=abs(o)).sum())/(Bn+1)
    return dict(B=Bn,seed=seed,obs_U=ob,obs_C=obc,mean_U=nu.mean(),sd_U=nu.std(ddof=1),p_U=pv(ob,nu),
                mean_C=ncu.mean(),sd_C=ncu.std(ddof=1),p_C=pv(obc,ncu),min_U=nu.min(),max_U=nu.max()), nu, ncu
t=time.time(); R['perm'], perm_U, perm_C = run_perm()
print(json.dumps(R['perm'],indent=1,default=float)); print('time %.0fs'%(time.time()-t))

## 8. Experiment 3 — degree-preserving randomization (double-edge swaps)
Each randomized graph starts from the observed graph and receives `SWAP_FACTOR × m` successful swaps (no self-loops or multi-edges). A separate single chain records clustering as swaps accumulate (mixing check). **This is the slow cell (~3–5 min).**

In [ ]:
def tc(H): return nx.transitivity(H), nx.average_clustering(H)
nsw = SWAP_FACTOR*m; rs = random.Random(SWAP_SEED); res=[]; trace=None; t=time.time()
for i in range(N_RANDOM):
    H=G.copy()
    nx.double_edge_swap(H, nswap=nsw, max_tries=nsw*100, seed=rs.randint(0,10**9))
    res.append(tc(H))
    if i==0:                                   # convergence trace on one chain
        H2=G.copy(); trace=[]
        for k in range(21):
            trace.append((k*m/2,)+tc(H2))
            nx.double_edge_swap(H2, nswap=int(m/2), max_tries=int(m/2)*100, seed=rs.randint(0,10**9))
res=np.array(res)
R['rand']=dict(Bg=len(res), nswap=nsw, ratio=nsw/m,
  T_mean=res[:,0].mean(), T_sd=res[:,0].std(ddof=1), T_min=res[:,0].min(), T_max=res[:,0].max(),
  C_mean=res[:,1].mean(), C_sd=res[:,1].std(ddof=1), C_min=res[:,1].min(), C_max=res[:,1].max(),
  zT=(R['transitivity']-res[:,0].mean())/res[:,0].std(ddof=1),
  zC=(R['avg_clust']-res[:,1].mean())/res[:,1].std(ddof=1),
  pT=(1+(res[:,0]>=R['transitivity']).sum())/(len(res)+1),
  pC=(1+(res[:,1]>=R['avg_clust']).sum())/(len(res)+1))
R['trace']=trace
k_=deg.astype(float); R['conf_expect']=((k_**2).mean()-k_.mean())**2/(n*k_.mean()**3)  # analytic approximation
print(json.dumps(R['rand'],indent=1,default=float)); print('analytic config-model approx:',R['conf_expect'])
print('time %.0fs'%(time.time()-t))

## 9. Figures (saved as PDF and shown inline)

In [ ]:
os.makedirs('figs',exist_ok=True)
# F1 degree distribution
fig,ax=plt.subplots(1,2,figsize=(7,2.7))
ax[0].hist(deg,bins=40,color='#4C72B0',edgecolor='white'); ax[0].set_xlabel('Degree $k$'); ax[0].set_ylabel('Number of nodes')
dd_=deg[deg>0]; ax[1].hist(dd_,bins=np.logspace(0,np.log10(dd_.max()),30),color='#55A868',edgecolor='white')
ax[1].set_xscale('log'); ax[1].set_yscale('log'); ax[1].set_xlabel('Degree $k$ (log)'); ax[1].set_ylabel('Number of nodes (log)')
plt.tight_layout(); plt.savefig('figs/f1_degree.pdf'); plt.show()

# F2 local clustering
fig,ax=plt.subplots(figsize=(3.4,2.6))
ax.hist(cc_arr[deg>=2],bins=40,color='#C44E52',edgecolor='white',label=r'$k_i\geq2$')
ax.hist(cc_arr[deg<2],bins=40,color='#999999',edgecolor='white',label=r'$k_i<2$ (set to 0)')
ax.axvline(R['avg_clust'],color='k',ls='--',lw=1,label=f"mean over all nodes = {R['avg_clust']:.3f}")
ax.set_xlabel('Local clustering $C_i$'); ax.set_ylabel('Number of nodes'); ax.legend(fontsize=7,frameon=False)
plt.tight_layout(); plt.savefig('figs/f2_localcc.pdf'); plt.show()

# F3 department closure under both estimands
fig,ax=plt.subplots(figsize=(3.4,2.6)); x=np.arange(2); wd=.36
U=[R['U_same'],R['U_diff']]; Cc=[R['C_same'],R['C_diff']]
ax.bar(x-wd/2,U,wd,label='Uniform over wedges',color='#4C72B0'); ax.bar(x+wd/2,Cc,wd,label='Uniform over centres',color='#DD8452')
for xi,v in zip(x-wd/2,U): ax.text(xi,v+.01,f'{v:.3f}',ha='center',fontsize=7)
for xi,v in zip(x+wd/2,Cc): ax.text(xi,v+.01,f'{v:.3f}',ha='center',fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(['Same dept.','Different dept.']); ax.set_ylabel('Wedge closure proportion'); ax.set_ylim(0,.9); ax.legend(fontsize=7,frameon=False)
plt.tight_layout(); plt.savefig('figs/f3_dept.pdf'); plt.show()

# F4 permutation null
fig,ax=plt.subplots(figsize=(3.4,2.6))
ax.hist(perm_U,bins=40,color='#8172B2',edgecolor='white'); ax.axvline(R['perm']['obs_U'],color='r',lw=1.5,label=f"observed = {R['perm']['obs_U']:.3f}")
ax.set_xlim(-.06,.45); ax.set_xlabel('Closure difference (same $-$ different)'); ax.set_ylabel('Permutations'); ax.legend(fontsize=7,frameon=False)
plt.tight_layout(); plt.savefig('figs/f4_perm.pdf'); plt.show()

# F5 randomized vs observed + convergence trace
fig,ax=plt.subplots(1,2,figsize=(7,2.7))
ax[0].hist(res[:,0],bins=15,alpha=.8,color='#4C72B0',label='Transitivity (swapped)'); ax[0].hist(res[:,1],bins=15,alpha=.8,color='#55A868',label='Avg. local clust. (swapped)')
ax[0].axvline(R['transitivity'],color='#4C72B0',ls='--'); ax[0].axvline(R['avg_clust'],color='#55A868',ls='--')
ax[0].set_xlabel('Clustering'); ax[0].set_ylabel('Randomised graphs'); ax[0].legend(fontsize=6.5,frameon=False,loc='center left'); ax[0].set_xlim(.1,.43)
tr=np.array(R['trace']); ax[1].plot(tr[:,0]/m,tr[:,1],'o-',ms=3,label='Transitivity'); ax[1].plot(tr[:,0]/m,tr[:,2],'s-',ms=3,label='Avg. local clust.')
ax[1].set_xlabel('Edge swaps performed / $m$'); ax[1].set_ylabel('Clustering (single chain)'); ax[1].legend(fontsize=7,frameon=False)
plt.tight_layout(); plt.savefig('figs/f5_random.pdf'); plt.show()

# F6 stratified closure
fig,ax=plt.subplots(1,2,figsize=(7,2.7))
for a,key,t in zip(ax,['strat_center','strat_minouter'],['Centre degree $k_B$','Smaller outer-node degree $\\min(k_A,k_C)$']):
    s=R[key]; g_=[x['group'] for x in s]; xx=np.arange(len(g_))
    a.bar(xx-.2,[x['same_rate'] for x in s],.4,label='Same dept.',color='#4C72B0'); a.bar(xx+.2,[x['diff_rate'] for x in s],.4,label='Different dept.',color='#DD8452')
    a.set_xticks(xx); a.set_xticklabels(g_); a.set_xlabel(t); a.set_ylabel('Closure proportion'); a.legend(fontsize=7,frameon=False)
plt.tight_layout(); plt.savefig('figs/f6_strat.pdf'); plt.show()

## 10. Save results and download

In [ ]:
json.dump(R, open('results.json','w'), indent=1, default=float)
np.save('perm_U.npy',perm_U); np.save('rand.npy',res)
try:
    import shutil; shutil.make_archive('triadic_outputs','zip','.','figs')
    from google.colab import files; files.download('results.json'); files.download('triadic_outputs.zip')
except ImportError:
    pass
print('Saved results.json and figs/')

In [3]:

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [4]:

!git --version



git version 2.43.0


In [5]:

import subprocess

GITHUB_USERNAME = "himakshimann"
REPO_NAME = "research-paper-implementation"

subprocess.run(
    ["git", "config", "--global", "user.name", GITHUB_USERNAME]
)

# Use the email associated with your GitHub account
subprocess.run([
    "git", "config", "--global", "user.email",
    "himakshimann@gmail.com"
])

print("Git configured successfully!")


Git configured successfully!


In [6]:

import os

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

os.chdir('/content')

!git clone {REPO_URL}

%cd /content/research-paper-implementation


Cloning into 'research-paper-implementation'...
/content/research-paper-implementation


In [ ]:

import shutil
import os


NOTEBOOK_PATH = "/content/drive/MyDrive/Colab Notebooks/Research_Paper_Implementation.ipynb"

# Copy notebook into the GitHub repository
shutil.copy(
    NOTEBOOK_PATH,
    "/content/research-paper-implementation/Research_Paper_Implementation.ipynb"
)

print("Notebook copied successfully!")
